# Dimensiones de la red multicapa

Nodos y enlaces de las tres capas, disponibilidad de datos por especie y
distribución de grado de las afiliaciones. Son las cifras que se citan en el
texto (Tables 1–3 del paper 2019 y S1 Fig del 2016).

Fuente: `analiceDB/01_tablas_analisis.ipynb` de v3 + `reproducir_v6/descriptivos_red/`.

Independiente del barrido: se puede correr en cualquier momento (README §6.4).
Salidas: `01_dimensiones_capas.csv`, `01_disponibilidad_especie.csv`,
`01_grado_afiliaciones.csv`.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

V4 = "/home/ggiordano/TDR/TDR_2026_v4"
sys.path.insert(0, f"{V4}/comun")
sys.path.insert(0, f"{V4}/analiceDB")
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_analice as fa              # el .py de esta carpeta

SALIDAS = tdr.out("analiceDB")
FIGURAS = SALIDAS / "figuras"
NB      = "01"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
# cluster_consistent=False: acá se describe la base entera, sin filtrar nada.
datos = tdr.cargar_db(anotaciones=True, fenotipo=True, cluster_consistent=False)
datos.st.shape, datos.bioact.shape, datos.sta.shape

## Acondicionamiento

In [ ]:
spoi = datos.especies_con_druggables(minimo=10)
resumen_tablas = pd.DataFrame({
    "tabla": ["00_specie_target", "03_bioactivities_target_compound",
              "03_bioactivities_organism_compound", "04a+04b (sta)"],
    "filas": [len(datos.st), len(datos.bioact), len(datos.bioact_org), len(datos.sta)],
})
print(f"{len(spoi)} especies con más de 10 druggables")
resumen_tablas

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
dim  = fa.dimensiones_capas(datos)
disp = fa.disponibilidad_por_especie(datos)
grad = fa.grado_afiliaciones(datos)

dim.to_csv(SALIDAS / f"{NB}_dimensiones_capas.csv", index=False)
disp.to_csv(SALIDAS / f"{NB}_disponibilidad_especie.csv", index=False)
grad.to_csv(SALIDAS / f"{NB}_grado_afiliaciones.csv", index=False)
fa.escribir_meta(SALIDAS, NB, notebook="01_dimensiones_red.ipynb",
                 params={"cluster_consistent": False, "con_quimica": False}, especies=spoi)

# Resultados

In [ ]:
dim  = pd.read_csv(SALIDAS / f"{NB}_dimensiones_capas.csv")
disp = pd.read_csv(SALIDAS / f"{NB}_disponibilidad_especie.csv")
grad = pd.read_csv(SALIDAS / f"{NB}_grado_afiliaciones.csv")
dim

In [ ]:
disp[disp["sp_id"].isin(tdr.ESPECIES_16)][
    ["sp_id", "nombre", "grupo", "proteinas", "con_interpro", "con_orthomcl",
     "pct_interpro", "pct_orthomcl", "druggables"]].round(1)

In [ ]:
fig = fa.fig_disponibilidad(disp, plt)
tdr.guardar(fig, f"{NB}_f01_disponibilidad", FIGURAS)

In [ ]:
# Cola pesada: las categorías grandes son las que beta > 0 (G'rk) atenúa
fig = fa.fig_grado(grad, plt)
tdr.guardar(fig, f"{NB}_f02_grado_afiliaciones", FIGURAS)

In [ ]:
t

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5), tight_layout=True)
t = grad.groupby(["tramo", "db"], observed=True).size().unstack(fill_value=0)
t['inicio'] = t.index
t['inicio'] = t['inicio'].apply(lambda x: x.split('-')[0].lstrip(">")).astype(int)
t.sort_values('inicio', inplace=True)
x = np.arange(len(t))
ax.bar(x - 0.2, t.get("ip", 0), width=0.4, color=tdr.S1, label="InterPro")
ax.bar(x + 0.2, t.get("omcl", 0), width=0.4, color=tdr.S2, label="OrthoMCL")
ax.set_xticks(x); ax.set_xticklabels(t.index, fontsize=8)
ax.set_yscale("log")
ax.set_xlabel("proteínas anotadas a la categoría")
ax.set_ylabel("categorías")
ax.legend(fontsize=8)
tdr.guardar(fig, f"{NB}_f03_tramos_de_grado", FIGURAS)